In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from sklearn.cluster import KMeans


In [ ]:
hourly = pd.read_csv('../01_Data/processed/hourly_energy.csv', parse_dates=['date_time'])


In [ ]:
hourly.head()


## Making daily profiles

In [ ]:
profiles = hourly.pivot_table(index=hourly["date_time"].dt.date, columns="hour", values="energy_kwh", aggfunc="mean")


In [ ]:
profiles = profiles.dropna()


In [ ]:
profiles.shape


In [ ]:
scaled = profiles.div(profiles.mean(axis=1), axis=0)


## Choosing number of clusters

In [ ]:
for k in [2,3,4,5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(scaled)
    print(k, round(model.inertia_, 2))


## Three clusters

In [ ]:
model = KMeans(n_clusters=3, random_state=42, n_init=10)


In [ ]:
labels = model.fit_predict(scaled)


In [ ]:
profiles["cluster"] = labels


In [ ]:
profiles["cluster"].value_counts().sort_index()


In [ ]:
centers = pd.DataFrame(model.cluster_centers_, columns=range(24))


In [ ]:
centers


## Cluster graph

In [ ]:
plt.figure(figsize=(10,4))
for i in range(3):
    plt.plot(centers.loc[i], label="Cluster " + str(i))
plt.xlabel("Hour")
plt.ylabel("Relative load")
plt.title("Daily Energy Usage Patterns")
plt.legend()
plt.show()


In [ ]:
plt.savefig('../04_Visuals/clustering/11_cluster_profiles.png', dpi=150)
plt.show()


In [ ]:
profiles.index.name = "day"


In [ ]:
profiles.reset_index().to_csv('../01_Data/processed/daily_load_profiles.csv', index=False)
